# 🎓 المساعد الجامعي الذكي — تدريب وفهرسة على Colab (GPU)

| # | المرحلة | ما يحدث |
|---|---|---|
| 1 | تجهيز البيانات | إطار كل ثانيتين من فيديوهاتك ← حذف المتكرر ← فرز أولي إلى `slide / blank / transition` |
| 2 | **تدريب YOLOv8n-cls** | ضبط دقيق + Top-1 / Top-5 / Confusion Matrix على val وtest |
| 3 | (اختياري) فهرسة المحاضرات | PaddleOCR + Whisper + Qwen + ChromaDB على نفس الفيديوهات |

> ⚠️ **ما يُدرَّب فعلاً هو YOLOv8n-cls فقط.** Whisper وPaddleOCR وQwen نماذج جاهزة تُستخدم دون تدريب.

## 📁 ما تجهّزه في Google Drive
```
MyDrive/UniversityAI/
├── LectureEngine.zip          ← حزمة المشروع (الملف الذي أرسلته لك)
├── videos/                    ← فيديوهات المحاضرات (3 على الأقل، والأفضل 8–10)
│     └── (اختياري) videos.txt  روابط يوتيوب/درايف، سطر لكل رابط
├── manual/                    ← (اختياري) صور موسومة يدوياً تدعم الفئات الشحيحة
│     ├── slide/   ├── blank/   └── transition/
└── outputs/                   ← يُنشأ تلقائياً: best.pt + التقارير + الفهارس
```
## ▶️ التشغيل
1. `Runtime ← Change runtime type ← T4 GPU` ثم **Save**.
2. اضبط الخلية 2 ثم `Runtime ← Run all`. لا تغلق التبويب.
3. للمراجعة اليدوية قبل التدريب: اضبط `STAGE = "prepare"` ← راجع مجلد `UniversityAI/review` في Drive ← أعد التشغيل بـ `STAGE = "train"`.

## 1) فحص الـ GPU

In [ ]:
import subprocess, shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("⚠️ لا يوجد GPU: من القائمة Runtime ← Change runtime type ← T4 GPU. (سيعمل كل شيء على CPU لكن ببطء شديد)")

## 2) الإعدادات

In [ ]:
#@title الإعدادات
DRIVE_DIR = "/content/drive/MyDrive/UniversityAI"  #@param {type:"string"}
STAGE     = "all"        #@param ["all", "prepare", "train"]
EPOCHS    = 0            #@param {type:"integer"}
MIN_TOP1  = 0.90         #@param {type:"number"}
FORCE_REPREP = False     #@param {type:"boolean"}
DO_INDEX  = True         #@param {type:"boolean"}
ASR_MODEL = "large-v3"   #@param ["large-v3", "medium", "small"]
LANGUAGE  = "ar"         #@param ["ar", "en", "auto"]
QWEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"  #@param ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]

## 3) ربط Drive والتحقق من الملفات

In [ ]:
import os, sys, shutil, subprocess, zipfile, json
from pathlib import Path

READY = True
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("(خارج Colab أو تعذر ربط Drive)")

D = Path(DRIVE_DIR)
for sub in ("videos", "manual/slide", "manual/blank", "manual/transition", "outputs"):
    (D / sub).mkdir(parents=True, exist_ok=True)

VIDEO_EXT = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v", ".flv", ".wmv"}
videos = sorted(p for p in (D / "videos").rglob("*") if p.suffix.lower() in VIDEO_EXT)
url_file = D / "videos" / "videos.txt"
has_urls = url_file.exists() and any(l.strip() and not l.startswith("#") for l in url_file.read_text(encoding="utf-8").splitlines())
ZIP = D / "LectureEngine.zip"

if not ZIP.exists():
    READY = False
    print(f"⛔ ضع حزمة المشروع هنا ثم أعد تشغيل الخلية: {ZIP}")
elif not videos and not has_urls:
    READY = False
    print(f"⛔ لا توجد فيديوهات. ضعها في: {D / 'videos'}")
else:
    total_gb = sum(p.stat().st_size for p in videos) / 1e9
    print(f"✅ {len(videos)} فيديو ({total_gb:.2f} GB)" + (" + روابط في videos.txt" if has_urls else ""))
    for p in videos:
        print(f"   - {p.name} ({p.stat().st_size / 1e6:.0f} MB)")
    if 0 < len(videos) < 3:
        print("⚠️ أقل من 3 فيديوهات: سيُستخدم تقسيم زمني (تقييم أضعف). الأفضل 3 فأكثر.")

## 4) تجهيز المشروع محلياً ونسخ الفيديوهات لقرص Colab (أسرع من Drive)

In [ ]:
def run(cmd, cwd=None, env=None, quiet=False):
    """يشغّل أمراً ويعرض مخرجاته مباشرة؛ يعيد رمز الخروج (لا يرفع استثناءً)."""
    e = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=e, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail.append(line)
        if not quiet:
            print(line, end="")
    rc = p.wait()
    if quiet and rc != 0:
        print("".join(tail[-15:]))
    return rc

PROJ, WORK, OUTL = Path("/content/LectureEngine"), Path("/content/work"), Path("/content/outputs")
LOCAL_VIDEOS = Path("/content/videos")
if READY:
    tmp = Path("/content/_unz"); shutil.rmtree(tmp, ignore_errors=True); shutil.rmtree(PROJ, ignore_errors=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(tmp)
    roots = [p for p in tmp.iterdir() if p.is_dir()]
    shutil.move(str(roots[0] if len(roots) == 1 else tmp), str(PROJ))
    OUTL.mkdir(exist_ok=True)

    shutil.rmtree(LOCAL_VIDEOS, ignore_errors=True)
    free_gb = shutil.disk_usage("/content").free / 1e9
    if videos and total_gb * 1.1 < free_gb:
        shutil.copytree(D / "videos", LOCAL_VIDEOS)
        VIDEOS_DIR = LOCAL_VIDEOS
    else:
        VIDEOS_DIR = D / "videos"
    VIDEO_ARGS = [str(VIDEOS_DIR)] + ([str(url_file)] if has_urls else [])

    # استعادة مراجعة سابقة من Drive (مراجعتك اليدوية محفوظة بين الجلسات)
    if FORCE_REPREP:
        shutil.rmtree(D / "review", ignore_errors=True)
    if (D / "review").exists() and any((D / "review").rglob("*.jpg")):
        shutil.rmtree(WORK / "review", ignore_errors=True)
        shutil.copytree(D / "review", WORK / "review")
        print("↩️ استُعيد مجلد review من Drive (سيُستخدم كما هو دون إعادة استخراج)")
    print("✅ جاهز | الفيديوهات:", VIDEOS_DIR)

## 5) تثبيت المكتبات (عدة دقائق)
تُنفَّذ كل المراحل الثقيلة في عمليات منفصلة، فلا حاجة لإعادة تشغيل الجلسة بعد التثبيت.

In [ ]:
if READY:
    ok = False
    for attempt in (1, 2):
        rc = run([sys.executable, "-m", "pip", "install", "-q", "-r", PROJ / "requirements_colab.txt"], quiet=True)
        if rc == 0:
            ok = True
            break
        print(f"…إعادة محاولة التثبيت ({attempt})")
    print("✅ تم التثبيت" if ok else "⚠️ تعذر تثبيت بعض الحزم (راجع الرسائل أعلاه)؛ سيُكمل النظام بما توفّر.")
    print("\n--- فحص الجاهزية ---")
    run([sys.executable, "scripts/selftest.py"], cwd=PROJ, env={"PYTHONPATH": str(PROJ / "src")})

## 6) التدريب 🚀
`STAGE="all"`: تجهيز ← تقسيم ← تدريب ← تقييم. أو `prepare` لإيقاف التنفيذ بعد الفرز الأولي لتراجع الصور في Drive.

In [ ]:
TRAIN_RC = None
if READY:
    cmd = [sys.executable, "training/run_training.py", "--videos", *VIDEO_ARGS,
           "--manual", D / "manual", "--work", WORK, "--out", OUTL, "--stage", STAGE,
           "--min-top1", MIN_TOP1]
    if EPOCHS and EPOCHS > 0:
        cmd += ["--epochs", EPOCHS]
    TRAIN_RC = run(cmd, cwd=PROJ)

    # احفظ فوراً في Drive (حتى لو انقطعت الجلسة لاحقاً)
    if (WORK / "review").exists():
        shutil.copytree(WORK / "review", D / "review", dirs_exist_ok=True)
        print(f"\n💾 مجلد المراجعة في Drive: {D / 'review'}  (انقل/احذف الصور الخاطئة ثم STAGE=train)")
    for sub in ("models", "reports"):
        if (OUTL / sub).exists():
            shutil.copytree(OUTL / sub, D / "outputs" / sub, dirs_exist_ok=True)
    if (WORK / "dataset_report.json").exists():
        shutil.copy2(WORK / "dataset_report.json", D / "outputs" / "reports" / "dataset_report.json") if (D / "outputs" / "reports").exists() else None
    print("\nرمز الخروج:", TRAIN_RC, "(0 = نجاح)")

## 7) النتائج 📊

In [ ]:
if READY:
    try:
        from IPython.display import Image, display
        m = json.loads((OUTL / "reports" / "metrics.json").read_text(encoding="utf-8"))
        for sp in ("val", "test"):
            r = m.get(sp, {})
            if r.get("n"):
                print(f"{sp}: Top-1 = {r['top1']:.3f} | Top-5 = {r['top5']:.3f} | n = {r['n']}")
        print("الهدف:", m["min_top1"], "| تحقق:", "✅" if m["passed"] else "⚠️ أقل من الهدف: راجع review وأضف بيانات")
        for sp in ("test", "val"):
            f = OUTL / "reports" / f"confusion_matrix_{sp}.png"
            if f.exists():
                display(Image(str(f)))
                break
        rep = json.loads((WORK / "dataset_report.json").read_text(encoding="utf-8"))
        print("\nالتقسيم:", rep["strategy"], rep["counts"])
        for w in rep.get("warnings", []):
            print("⚠️", w)
    except Exception as e:
        print("لا توجد نتائج تدريب بعد:", str(e)[:100])

## 8) (اختياري) فهرسة المحاضرات: PaddleOCR + Whisper + Qwen + ChromaDB

In [ ]:
if READY and DO_INDEX and STAGE != "prepare":
    best = OUTL / "models" / "best.pt"
    env = {"PYTHONPATH": str(PROJ / "src"), "STORAGE_DIR": str(OUTL / "storage"),
           "YOLO_CLS_WEIGHTS": str(best if best.exists() else D / "outputs" / "models" / "best.pt"),
           "ASR_MODEL": ASR_MODEL, "ASR_LANGUAGE": LANGUAGE, "QWEN_MODEL": QWEN_MODEL, "QWEN_BACKEND": "auto"}
    srcs = [str(p) for p in sorted(VIDEOS_DIR.rglob("*")) if p.suffix.lower() in VIDEO_EXT]
    if has_urls:
        srcs += [l.strip() for l in url_file.read_text(encoding="utf-8").splitlines() if l.strip() and not l.startswith("#")]
    for s in srcs:
        run([sys.executable, "scripts/process_video.py", s], cwd=PROJ, env=env)
    # احفظ الفهرس في Drive
    if (OUTL / "storage").exists():
        shutil.copytree(OUTL / "storage", D / "outputs" / "storage", dirs_exist_ok=True)
        print("💾 حُفظ الفهرس في", D / "outputs" / "storage")
else:
    print("تخطي الفهرسة (DO_INDEX=False أو STAGE=prepare)")

## 9) تنزيل النموذج 💾

In [ ]:
if READY:
    model = D / "outputs" / "models" / "best.pt"
    if not model.exists() and (OUTL / "models" / "best.pt").exists():
        model = OUTL / "models" / "best.pt"
    print("✅ ملف النموذج:" if model.exists() else "ℹ️ لا يوجد نموذج بعد:", model)
    try:
        from google.colab import files
        if model.exists():
            files.download(str(model))
    except Exception:
        pass

## ماذا بعد؟ 🧭
على جهازك/الخادم: انسخ `outputs/models/best.pt` إلى `LectureEngine/models/best.pt`، (وإن فهرست هنا) انسخ `outputs/storage` إلى `LectureEngine/storage`، ثم:
```bash
cp .env.example .env
bash scripts/run_server.sh        # http://localhost:8000/docs
```
التفاصيل الكاملة في `README.md`.

**إن ساءت النتائج:** افتح `UniversityAI/review` وصحّح الصور الخاطئة (خاصة `transition`)، أضف صوراً يدوية في `manual/`، ثم `STAGE="train"`.